In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# papermill injects overrides into this cell
# fixes the test hold-out AND the Dirichlet partition
split_seed = 42
# repetition seed: federated training AND per-silo generation stochasticity
train_seed = 42
num_participants = 30
num_rounds = 5
# 1.0 = use the full train pool; explicit and recorded
scarcity = 1.0
output_dir = "results/run"
csv_output_path = ""


In [ ]:
import os
os.environ["FL_BRANCH"] = "baseline"
os.environ["FL_SPLIT_SEED"] = str(split_seed)
os.environ["FL_TRAIN_SEED"] = str(train_seed)
os.environ["FL_NUM_PARTICIPANTS"] = str(num_participants)
os.environ["FL_NUM_ROUNDS"] = str(num_rounds)
os.environ["FL_SCARCITY"] = str(scarcity)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf

# Keep the FL classifier on CPU (as before) WITHOUT hiding the GPU from torch:
# the per-silo GANs are torch and must stay on the GPU. Setting
# CUDA_VISIBLE_DEVICES=-1 here, as the previous version did, would force
# generator training onto CPU and make the grid untenable.
tf.config.set_visible_devices([], "GPU")
print("TensorFlow restricted to CPU; torch retains GPU access")


In [ ]:
from model_test_dataset1 import modello_test
from federated_averaging import federated_averaging
#from generate_random_weights import generate_random_weights
#from evaluate_model_with_mixed_weights import evaluate_model_with_mixed_weights

In [ ]:
import syft as sy

# Launch a fresh domain server named "test-domain-1" in dev mode on the local machine
port = 3331
node = sy.orchestra.launch(name="test-domain-1", port=port, dev_mode=True, reset=True)
domain_client = sy.login(port=port, email="info@openmined.org", password="changethis")


In [ ]:
# Register a Data Scientist 
domain_client.register(name = "DS", email = "DS@caltech.edu", password="abc123", password_verify="abc123")
DS = domain_client.login(email="DS@caltech.edu", password="abc123")

In [ ]:
%run new_users.py

# Uses the variables created by the data-prep script above.
print(f"\nClients: {len(client_datasets)}")
print(f"Features: {client_datasets[0][0].shape[1] if client_datasets[0][0].ndim == 2 else client_datasets[0][0].shape[1:]}")
print(f"Global held-out test set: {len(y_test_global)} samples")

# Aliases kept for the cells below. The test set is the SAME for every client.
test_X = X_test_global
test_y = y_test_global


In [ ]:
# Run all scripts in the 'scripts' folder
import os
scripts_dir = 'scripts'
for script_file in os.listdir(scripts_dir):
    if script_file.endswith('.py'):
        script_path = os.path.join(scripts_dir, script_file)
        %run $script_path


In [ ]:
@sy.syft_function()
def train_fn(X_train, Y_train):
    import tensorflow as tf
    from tensorflow.keras import layers, models
    import numpy as np

    # Convert to NumPy array if needed
    X_train = np.array(X_train)
    Y_train = np.array(Y_train)
    
    # Define the deeper model (5 layers)
    input_dim = X_train.shape[1]
    
    model = models.Sequential([
        # Layer 1
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        #layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Layer 2
        layers.Dense(32, activation='relu'),
        #layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Layer 3
        layers.Dense(16, activation='relu'),
        #layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        # Layer 4
        layers.Dense(8, activation='relu'),
        layers.Dropout(0.4),
        
        # Output layer
        layers.Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # Train the model (1 epoch per FL round)
    model.fit(X_train, Y_train, epochs=1, batch_size=32)
    
    # Get the model weights
    model_weights = model.get_weights()
    
    return model, model_weights

In [ ]:
nuovo_progetto = sy.Project(name = 'Training della rete MNIST', members = [DS])
nuovo_progetto

In [ ]:
nuovo_progetto.create_code_request(train_fn,DS)

In [ ]:
DS.code;

In [ ]:
progetto = nuovo_progetto.start()
progetto.events

In [ ]:
progetto.requests;

In [ ]:
progetto.requests[0];

In [ ]:
domain_client.projects

In [ ]:
requests = domain_client.projects[0].requests
requests

In [ ]:
from syft.service.request.request import RequestStatus
request = requests[0]
assert request.status == RequestStatus.PENDING
request;

In [ ]:
change = request.changes[0]
change

In [ ]:
user_code = change.link
user_code.show_code

In [ ]:
user_code.assets

In [ ]:
user_function = user_code.unsafe_function
user_function

# START OF MULTI-USER FEDERATED LEARNING

In [ ]:
# =========================================================
# SEED FEDERATED TRAINING
# =========================================================
# train_seed must actually control training stochasticity, otherwise the three
# "repetitions" differ only by TensorFlow's unseeded default RNG: runs are then
# irreproducible and the train_seed recorded in the manifest is a label with no
# causal connection to the result.
#
# Seeded HERE, after data preparation, so train_seed cannot influence the test
# split, the Dirichlet partition, or generation -- those are governed by
# split_seed alone (fl_core/config.py).
import os, random
import numpy as _np
import tensorflow as _tf

_TRAIN_SEED = int(os.environ.get("FL_TRAIN_SEED", 42))
random.seed(_TRAIN_SEED)
_np.random.seed(_TRAIN_SEED)
_tf.random.set_seed(_TRAIN_SEED)
print(f"Federated training seeded with train_seed={_TRAIN_SEED}")


In [ ]:
# Train all participant models — each participant corresponds to one script.
import os

# List to store results for each user
client_model_list = []
client_weights_list = []


# Run all scripts in the 'scripts' folder
scripts_dir = 'scripts'
for script_file in os.listdir(scripts_dir):
    if script_file.endswith('.py'):
        script_path = os.path.join(scripts_dir, script_file)
        
        # Get the user index from the script filename
        user_index = int(script_file.split('_')[-1].split('.')[0])
        
        # Run user_function for the current user using the corresponding data
        data_prefix = f"_a_{user_index}"
        modello_alice, modello_pesi_alice = user_function(X_train=eval(f"train_X{data_prefix}"), Y_train=eval(f"train_y{data_prefix}"))
        
        # Append results to the list
        client_model_list.append(modello_alice)
        client_weights_list.append(modello_pesi_alice)

# You can now access results for each user usingo le liste client_model_list e client_weights_list


In [ ]:
deposito_risultati = request.accept_by_depositing_result(client_weights_list)

# END OF MULTI-USER FEDERATED LEARNING

In [ ]:
global_model = federated_averaging(modello_test(), client_model_list)

In [ ]:
esempio_test = global_model.evaluate(test_X,test_y)

In [ ]:
# Compute precision, recall, F1-score and confusion matrix

from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get predictions from the global model
y_pred_prob = global_model.predict(test_X)
y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)  # Convert probabilities to binary classes

# Compute metrics
print("Evaluation metrics:")
print(f"Accuracy: {global_model.evaluate(test_X, test_y, verbose=0)[1]:.4f}")

# Precision, recall, F1-score per class
precision, recall, f1, _ = precision_recall_fscore_support(test_y, y_pred, average=None)
print("\nDetailed metrics per class:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"Class '{class_name}':")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")

# Full classification report
print("\nFull classification report:")
print(classification_report(test_y, y_pred, target_names=label_encoder.classes_))

# Confusion matrix visualisation
cm = confusion_matrix(test_y, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Real')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import sys
import io
import os
import gc
from datetime import datetime
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
import tensorflow as tf
import time
from tensorflow.keras import layers, models

# =========================================================
# FEDERATED LEARNING WITH MULTIPLE ROUNDS
# =========================================================
num_rounds = globals().get("num_rounds", 10)  # <-- change ONLY this and re-run the cell
EPOCHS_LOCAL = 1
BATCH_SIZE = 32
LR = 0.0005

print("=== STARTING FEDERATED LEARNING WITH MULTIPLE ROUNDS ===")

# =========================================================
# FORCED RESET (inherits nothing from the previous run)
# =========================================================
tf.keras.backend.clear_session()
gc.collect()

# clear_session() resets the Keras graph, so re-seed to keep weight init and
# batch shuffling deterministic for this train_seed.
import random as _random
_random.seed(_TRAIN_SEED)
np.random.seed(_TRAIN_SEED)
tf.random.set_seed(_TRAIN_SEED)



# =========================================================
# Count participants
# =========================================================
scripts_dir = 'scripts'
script_files = sorted([f for f in os.listdir(scripts_dir) if f.endswith('.py')])
num_participants = len(script_files)

# =========================================================
# Function: build model (always fresh)
# =========================================================
def build_model(input_dim, lr=LR):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.5),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(8, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# =========================================================
# Function: compute detailed metrics
# =========================================================
def calculate_detailed_metrics(model, X, y_true):
    y_pred_prob = model.predict(X, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    precision_avg, recall_avg, f1_avg, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    FP = cm.sum(axis=0) - np.diag(cm)
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    return {
        'precision_per_class': precision,
        'recall_per_class': recall,
        'f1_per_class': f1,
        'support_per_class': support,
        'precision_avg': precision_avg,
        'recall_avg': recall_avg,
        'f1_avg': f1_avg,
        'confusion_matrix': cm,
        'TP': TP,
        'TN': TN,
        'FP': FP,
        'FN': FN
    }

# =========================================================
# Initialise fresh global model (always new)
# =========================================================
input_dim = train_X.shape[1]
global_model = build_model(input_dim)

# Evaluate initial model
initial_acc = global_model.evaluate(test_X, test_y, verbose=0)[1]
print(f"Initial accuracy (fresh weights): {initial_acc:.4f}")

# =========================================================
# FL rounds
# =========================================================
global_accuracies = []
all_metrics = []
_timing_local_per_round = []
_timing_aggregation = []
_timing_local_total = 0.0

for round_num in range(num_rounds):
    print(f"\n--- ROUND {round_num + 1}/{num_rounds} ---")

    round_models = []
    _round_local_times = []

    for script_file in script_files:
        user_index = int(script_file.split('_')[-1].split('.')[0])
        data_prefix = f"_a_{user_index}"

        client_model = build_model(input_dim)
        client_model.set_weights(global_model.get_weights())

        train_X_local = eval(f"train_X{data_prefix}")
        train_y_local = eval(f"train_y{data_prefix}")

        print(f"  Client {user_index}: Local training...")
        _t0_local = time.perf_counter()
        client_model.fit(
            train_X_local, train_y_local,
            epochs=EPOCHS_LOCAL,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        _dt_local = time.perf_counter() - _t0_local
        _round_local_times.append(_dt_local)
        _timing_local_total += _dt_local
        round_models.append(client_model)

    _timing_local_per_round.append(np.mean(_round_local_times))
    print("  Running federated aggregation...")
    old_stdout = sys.stdout
    sys.stdout = buffer = io.StringIO()
    _t0_agg = time.perf_counter()
    global_model = federated_averaging(global_model, round_models)
    _timing_aggregation.append(time.perf_counter() - _t0_agg)
    sys.stdout = old_stdout

    global_acc = global_model.evaluate(test_X, test_y, verbose=0)[1]
    metrics = calculate_detailed_metrics(global_model, test_X, test_y)

    global_accuracies.append(global_acc)
    all_metrics.append(metrics)

    print(f"  Global accuracy: {global_acc:.4f}")
    print(f"  Precision (weighted): {metrics['precision_avg']:.4f}")
    print(f"  Recall (weighted): {metrics['recall_avg']:.4f}")
    print(f"  F1-Score (weighted): {metrics['f1_avg']:.4f}")
    print(f"  Total false positives: {metrics['FP'].sum()}")
    print(f"  Total false negatives: {metrics['FN'].sum()}")

# =========================================================
# TIMING SUMMARY
# =========================================================
_mean_local = np.mean(_timing_local_per_round)
_mean_agg = np.mean(_timing_aggregation)
_total_fl_time = _timing_local_total + sum(_timing_aggregation)

_timing_lines = [
    "=" * 80,
    "FL TIMING RESULTS",
    "=" * 80,
    f"Dataset          : CIC-MalMem2022",
    f"Configuration    : {num_participants} participants x {num_rounds} rounds",
    f"Mean local training time per round (avg across participants): {_mean_local:.6f} s",
    f"Mean aggregation time per round                             : {_mean_agg:.6f} s",
    f"Total FL time (local training + aggregation only)           : {_total_fl_time:.6f} s",
    "=" * 80,
]
_timing_summary = "\n".join(_timing_lines)
print("\n" + _timing_summary)

_timing_file = "fl_timing_results.txt"
with open(_timing_file, "a", encoding="utf-8") as _tf:
    _tf.write("\n" + _timing_summary + "\n")
print(f"Timing results appended to: {_timing_file}")

# =========================================================
# FINAL RESULTS + REPORT
# =========================================================
print(f"\n=== FINAL RESULTS ===")
final_metrics = all_metrics[-1]
print(f"Final accuracy: {global_accuracies[-1]:.4f}")
print(f"Final precision: {final_metrics['precision_avg']:.4f}")
print(f"Final recall: {final_metrics['recall_avg']:.4f}")
print(f"Final F1-score: {final_metrics['f1_avg']:.4f}")
print(f"\nAccuracy improvement: {global_accuracies[-1] - initial_acc:.4f}")

print(f"\n=== METRICS PER CLASS ===")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"\n{class_name}:")
    print(f"  Precision: {final_metrics['precision_per_class'][i]:.4f}")
    print(f"  Recall: {final_metrics['recall_per_class'][i]:.4f}")
    print(f"  F1-Score: {final_metrics['f1_per_class'][i]:.4f}")
    print(f"  TP: {final_metrics['TP'][i]}, TN: {final_metrics['TN'][i]}")
    print(f"  FP: {final_metrics['FP'][i]}, FN: {final_metrics['FN'][i]}")

print(f"\n=== CLASSIFICATION REPORT ===")
y_pred_final = (global_model.predict(test_X, verbose=0) > 0.5).astype(int).reshape(-1)
report_str = classification_report(test_y, y_pred_final, target_names=label_encoder.classes_)
print(report_str)

# =====================================================
# SAVE RESULTS TO TXT FILE
# =====================================================
output_dir = globals().get("output_dir", 'review_malware_results')
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"FL_results_p{num_participants}_r{num_rounds}_{timestamp}.txt"
txt_path = os.path.join(output_dir, filename)

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("FEDERATED LEARNING - RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write(f"Date and time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Number of participants: {num_participants}\n")
    f.write(f"Number of rounds: {num_rounds}\n")
    f.write(f"Classes: {', '.join(label_encoder.classes_)}\n")
    f.write(f"Dataset: MalMem2022\n")
    f.write("\n" + "=" * 80 + "\n\n")

    f.write("GLOBAL RESULTS\n")
    f.write("-" * 80 + "\n")
    f.write(f"Initial accuracy (fresh weights): {initial_acc:.4f}\n")
    f.write(f"Final accuracy: {global_accuracies[-1]:.4f}\n")
    f.write(f"Accuracy improvement: {global_accuracies[-1] - initial_acc:.4f}\n")
    f.write(f"Final precision (weighted): {final_metrics['precision_avg']:.4f}\n")
    f.write(f"Final recall (weighted): {final_metrics['recall_avg']:.4f}\n")
    f.write(f"Final F1-score (weighted): {final_metrics['f1_avg']:.4f}\n")
    f.write(f"Total false positives: {final_metrics['FP'].sum()}\n")
    f.write(f"Total false negatives: {final_metrics['FN'].sum()}\n")

    f.write("\n" + "=" * 80 + "\n\n")
    f.write("ROUND PROGRESSION\n")
    f.write("-" * 80 + "\n")
    for i in range(num_rounds):
        f.write(f"\nROUND {i + 1}/{num_rounds}:\n")
        f.write(f"  Accuracy: {global_accuracies[i]:.4f}\n")
        f.write(f"  Precision: {all_metrics[i]['precision_avg']:.4f}\n")
        f.write(f"  Recall: {all_metrics[i]['recall_avg']:.4f}\n")
        f.write(f"  F1-Score: {all_metrics[i]['f1_avg']:.4f}\n")
        f.write(f"  False Positives: {all_metrics[i]['FP'].sum()}\n")
        f.write(f"  False Negatives: {all_metrics[i]['FN'].sum()}\n")

    f.write("\n" + "=" * 80 + "\n\n")
    f.write("METRICS PER CLASS\n")
    f.write("-" * 80 + "\n")
    for i, class_name in enumerate(label_encoder.classes_):
        f.write(f"\n{class_name}:\n")
        f.write(f"  Precision: {final_metrics['precision_per_class'][i]:.4f}\n")
        f.write(f"  Recall: {final_metrics['recall_per_class'][i]:.4f}\n")
        f.write(f"  F1-Score: {final_metrics['f1_per_class'][i]:.4f}\n")
        f.write(f"  Support: {final_metrics['support_per_class'][i]}\n")
        f.write(f"  True Positives: {final_metrics['TP'][i]}\n")
        f.write(f"  True Negatives: {final_metrics['TN'][i]}\n")
        f.write(f"  False Positives: {final_metrics['FP'][i]}\n")
        f.write(f"  False Negatives: {final_metrics['FN'][i]}\n")

    f.write("\n" + "=" * 80 + "\n\n")
    f.write("CLASSIFICATION REPORT\n")
    f.write("-" * 80 + "\n")
    f.write(report_str)

    f.write("\n" + "=" * 80 + "\n\n")
    f.write("CONFUSION MATRIX\n")
    f.write("-" * 80 + "\n")
    f.write(np.array2string(final_metrics['confusion_matrix'], separator=', '))
    f.write("\n")

print(f"\nResults saved to: {txt_path}")
print(f"File saved in: {output_dir}/")

# CSV per-round convergence log
_csv_path = globals().get("csv_output_path", "")
if _csv_path:
    import csv as _csv_mod
    with open(_csv_path, 'w', newline='', encoding='utf-8') as _cf:
        _w = _csv_mod.writer(_cf)
        _w.writerow(['round', 'accuracy', 'precision', 'recall', 'f1', 'false_positives', 'false_negatives'])
        for _i in range(num_rounds):
            _w.writerow([
                _i + 1,
                round(float(global_accuracies[_i]), 6),
                round(float(all_metrics[_i]['precision_avg']), 6),
                round(float(all_metrics[_i]['recall_avg']), 6),
                round(float(all_metrics[_i]['f1_avg']), 6),
                int(all_metrics[_i]['FP'].sum()),
                int(all_metrics[_i]['FN'].sum()),
            ])
    print(f"Per-round CSV saved to: {_csv_path}")

In [ ]:
# =========================================================
# PERSIST THE RUN MANIFEST
# =========================================================
# The manifest is what makes a run auditable and the grid resumable: it carries
# the metrics plus the full data accounting (per-silo real/synthetic counts,
# realised augmentation ratio, test-set hash, feature count, git SHA).
# aggregate_results.py builds results/aggregated/all_runs_long.csv from these,
# which is the input the Wilcoxon analysis needs.
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

_prob = global_model.predict(test_X, verbose=0)
_pred = (_prob > 0.5).astype(int).reshape(-1) if _prob.shape[-1] == 1 else _prob.argmax(axis=1)
_p, _r, _f1, _ = precision_recall_fscore_support(
    test_y, _pred, average="macro", zero_division=0)

run_manifest.metrics = {
    "accuracy": float(accuracy_score(test_y, _pred)),
    "precision": float(_p),
    "recall": float(_r),
    "f1": float(_f1),
}
_path = run_manifest.save()
print(f"Manifest written: {_path}")
print(f"  accuracy={run_manifest.metrics['accuracy']:.4f} "
      f"f1={run_manifest.metrics['f1']:.4f}")
print(f"  real={run_manifest.total_real} synthetic={run_manifest.total_synthetic} "
      f"realised_ratio={run_manifest.aug_ratio_realised:.4f}")


In [ ]:
import gc
import tensorflow as tf
for _n in ["client_model_list", "client_weights_list", "round_models",
           "global_model", "client_model"]:
    globals().pop(_n, None)
tf.keras.backend.clear_session()
gc.collect()
print("Memory cleanup complete.")